In [ ]:
import pandas as pd
import numpy as np
import os
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix
 
train_df = pd.read_csv('train_preprocessed.csv')
test_df = pd.read_csv('test_preprocessed.csv')
 
X_train = train_df.drop('RainTomorrow', axis=1)
y_train = train_df['RainTomorrow']
X_test = test_df.drop('RainTomorrow', axis=1)
y_test = test_df['RainTomorrow']

X_tune = X_train.sample(n=30_000, random_state=42)
y_tune = y_train.loc[X_tune.index]
 
lr_pipeline = Pipeline([
    ('lr', LogisticRegression(
        solver='saga',
        max_iter=500,
        random_state=42
    ))
])
 
param_grid = {
    'lr__l1_ratio': [0.0, 0.5, 1.0],  
    'lr__C': [0.01, 0.1, 0.5, 1.0],
}
 
search = GridSearchCV(
    lr_pipeline,
    param_grid=param_grid,
    cv=3,
    scoring='f1',
    n_jobs=-1
)
search.fit(X_tune, y_tune)
 
results = pd.DataFrame(search.cv_results_)
cols = ['param_lr__l1_ratio', 'param_lr__C', 'mean_test_score', 'rank_test_score']
report_df = results[cols].sort_values(by='mean_test_score', ascending=False)
report_df.columns = ['l1_ratio', 'C', 'avg_F1-Score', 'rank']
print(report_df.to_string(index=False))
 
best_lr = search.best_estimator_
 
best_lr.fit(X_train, y_train)
 
y_pred = best_lr.predict(X_test)
y_pred_proba = best_lr.predict_proba(X_test)[:, 1]

acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc_score = roc_auc_score(y_test, y_pred_proba)
cm = confusion_matrix(y_test, y_pred)
 
print(f"Accuracy:  {acc:.4f}")
print(f"Precision: {prec:.4f}")
print(f"Recall:    {rec:.4f}")
print(f"F1 Score:  {f1:.4f}")
print(f"ROC AUC Score: {auc_score:.4f}")
 
print("\nConfusion Matrix:")
print(cm)

plt.figure(figsize=(5, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['No Rain', 'Rain'],
            yticklabels=['No Rain', 'Rain'])

plt.title('Regularized Regression Confusion Matrix')
plt.ylabel('Actual')
plt.xlabel('Predicted')

plt.tight_layout()
plt.savefig('confusion_matrix_regreg.png')
plt.close()

results_file = 'model_results.csv'

new_result = pd.DataFrame([{
    'Model': 'Regularized Regression',
    'Accuracy': acc,
    'Precision': prec,
    'Recall': rec,
    'F1': f1,
    'ROC_AUC': auc_score
}])

if os.path.exists(results_file):
    existing_df = pd.read_csv(results_file)
else:
    existing_df = pd.DataFrame(columns=['Model', 'Accuracy', 'Precision', 'Recall', 'F1', 'ROC_AUC'])

existing_df = existing_df[existing_df['Model'] != 'Regularized Regression']
updated_df = pd.concat([existing_df, new_result], ignore_index=True)
updated_df.to_csv(results_file, index=False)
print(updated_df)
 
feature_names = X_train.columns.tolist()
coefs = best_lr.named_steps['lr'].coef_[0]
best_l1_ratio = search.best_params_['lr__l1_ratio']
penalty_label = {0.0: 'L2 (Ridge)', 1.0: 'L1 (Lasso)'}.get(best_l1_ratio, f'ElasticNet (l1_ratio={best_l1_ratio})')
 
importance_df = pd.DataFrame({
    'feature': feature_names,
    'coefficient': coefs,
    'abs_coefficient': np.abs(coefs)
}).sort_values('abs_coefficient', ascending=False)
 
print(f"\nTop 15 most influential features (best penalty: {penalty_label}):")
print(importance_df.head(15).to_string(index=False))
 
if best_l1_ratio > 0:
    n_zero = (coefs == 0).sum()
    print(f"\nL1 sparsity: {n_zero} / {len(feature_names)} features zeroed out "
          f"({n_zero / len(feature_names) * 100:.1f}% eliminated)")
 